# Stage 2: BI Data Export and Tableau Integration

## 📚 Objectives
- Export processed RFM data and funnel metrics to CSV
- (Optional) Upload to AWS S3 for cloud storage
- Create Tableau dashboard for business visualization
- Implement AWS S3 client functions

## 📋 Agenda
1. Load processed data from Stage 1
2. Prepare dimensions and metrics for BI
3. Export to CSV files
4. (Optional) Upload to AWS S3
5. Connect to Tableau Desktop

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# TODO: Import AWS client from src if implementing cloud storage
# from src.data_pipeline.aws_client import AWSS3Client

In [ ]:
# TODO: Load RFM segments from Stage 1 output
# rfm_df = pd.read_csv('data/processed/rfm_segments.csv')
# print(f"Loaded {len(rfm_df)} customer records")

## 2. Prepare Data for BI (Fact and Dimension Tables)

In [ ]:
# TODO: Create dimension tables for Tableau
# - Customer Dimension: customer_id, segment, r_score, f_score, m_score
# - Date Dimension: date, month, quarter, year
# - Product Dimension: product_id, category, price

# TODO: Create fact tables for BI
# - Order Facts: order_id, customer_id, date, amount, etc.

## 3. Calculate Funnel Metrics

In [ ]:
# TODO: Calculate conversion funnel
# - Total Visitors
# - Add to Cart
# - Checkout
# - Payment Completed
# - Calculate step-to-step conversion rates

## 4. Export to CSV

In [ ]:
# TODO: Export dimension and fact tables to CSV
# rfm_df.to_csv('data/processed/rfm_for_tableau.csv', index=False)
# funnel_df.to_csv('data/processed/funnel_metrics.csv', index=False)

print("Data exported successfully!")

## 5. (Optional) Upload to AWS S3

## 5a. AWS Account Setup & S3 Configuration

### Step 1: Create AWS Account
1. Go to https://aws.amazon.com/
2. Click **"Create an AWS Account"** (top right)
3. Enter email address and create password
4. Choose **"Personal"** for account type
5. Enter billing information (free tier available for 12 months)
6. Verify phone number via SMS
7. Select **"Basic"** support plan (free)
8. Account activation takes ~5-15 minutes

### Step 2: Create S3 Bucket

**Via AWS Console (Easy):**
1. Log in to AWS Management Console (https://console.aws.amazon.com/)
2. Search for **"S3"** in the service search bar
3. Click **"Create bucket"** button
4. **Bucket name:** Enter unique name (e.g., `my-ecommerce-data-2025`)
   - Must be lowercase, alphanumeric + hyphens only
   - Must be globally unique across all AWS accounts
5. **Region:** Select closest to you (e.g., us-east-1)
6. **Block Public Access:** Keep all checked (secure by default)
7. Click **"Create bucket"**
8. Go to bucket → **Permissions tab** → **Bucket Policy** → Add policy to allow programmatic access

**Example bucket policy:**
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::YOUR_ACCOUNT_ID:user/YOUR_USERNAME"
            },
            "Action": "s3:*",
            "Resource": [
                "arn:aws:s3:::YOUR_BUCKET_NAME",
                "arn:aws:s3:::YOUR_BUCKET_NAME/*"
            ]
        }
    ]
}
```

### Step 3: Create IAM User for Programmatic Access

1. Go to **IAM Service** (search in AWS console)
2. Click **"Users"** → **"Create user"**
3. **Username:** e.g., `ecommerce-app-user`
4. Check **"Provide user access to the AWS Management Console"** (optional)
5. Click **"Next"**
6. Select **"Attach policies directly"**
7. Search for **"AmazonS3FullAccess"** and check it
8. Click **"Create user"**
9. Click on user → **"Security credentials"** tab
10. **Create access key:**
    - Purpose: "Application running outside AWS"
    - Click **"Create access key"**
    - **IMPORTANT:** Save the Access Key ID and Secret Access Key (show only once!)

### Step 4: Connect to S3 from Python Code

**Method A: Using Environment Variables (Recommended - Most Secure)**

1. **On Windows:**
   - Press `Win + R`, type `SystemPropertiesAdvanced`, press Enter
   - Click **"Environment Variables"**
   - Under "User variables", click **"New"**
   - Variable name: `AWS_ACCESS_KEY_ID`
   - Variable value: (paste your Access Key ID)
   - Click OK, repeat for `AWS_SECRET_ACCESS_KEY`
   - **Restart your terminal/Python** for changes to take effect

2. **On Mac/Linux:**
   ```bash
   export AWS_ACCESS_KEY_ID=your_access_key_here
   export AWS_SECRET_ACCESS_KEY=your_secret_key_here
   ```
   Or add to `~/.bashrc` or `~/.zshrc` for persistence

**Method B: Using Credentials File**

1. Create file `~/.aws/credentials`:
   ```
   [default]
   aws_access_key_id = YOUR_ACCESS_KEY_ID
   aws_secret_access_key = YOUR_SECRET_ACCESS_KEY
   ```

2. Create file `~/.aws/config`:
   ```
   [default]
   region = us-east-1
   ```

**Method C: In Code (Less Secure - Only for Testing)**
```python
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id='YOUR_ACCESS_KEY',
    aws_secret_access_key='YOUR_SECRET_KEY',
    region_name='us-east-1'
)
```

### Step 5: Test Connection in Python

Run this cell to verify everything works:


In [ ]:
import boto3

# Test AWS connection
try:
    # Initialize S3 client (reads credentials from environment variables)
    s3 = boto3.client('s3')
    
    # List all buckets to verify connection
    response = s3.list_buckets()
    buckets = response['Buckets']
    
    print("✅ AWS Connection Successful!")
    print(f"Found {len(buckets)} bucket(s):")
    for bucket in buckets:
        print(f"  - {bucket['Name']}")
        
except Exception as e:
    print(f"❌ AWS Connection Failed: {str(e)}")
    print("Check your credentials are set correctly in environment variables")

In [ ]:
# TODO: Test AWS S3 upload functionality
# s3_client = AWSS3Client(bucket_name='my-bucket')
# s3_client.upload_dataframe_to_s3(rfm_df, 'data/rfm_segments.csv')
# s3_client.upload_dataframe_to_s3(funnel_df, 'data/funnel_metrics.csv')


### 📌 Implementation Task: Complete AWS S3 Client

**After testing S3 uploads in this notebook, implement in:** `src/data_pipeline/aws_client.py`

Complete these methods in the `AWSS3Client` class:

1. `__init__(bucket_name, aws_access_key_id=None, aws_secret_access_key=None)`
   - Initialize `boto3.client('s3')` with provided or environment credentials
   - Store bucket_name as instance variable

2. `upload_dataframe_to_s3(df, file_key)`
   - Convert DataFrame to CSV format (BytesIO stream)
   - Upload to S3 using `s3.put_object()`
   - Return True on success, False on failure
   - Example: `upload_dataframe_to_s3(rfm_df, 'data/rfm_segments.csv')`

3. `download_dataframe_from_s3(file_key)`
   - Download CSV from S3 using `s3.get_object()`
   - Parse with `pd.read_csv()`
   - Return DataFrame
   - Example: `rfm_df = s3_client.download_dataframe_from_s3('data/rfm_segments.csv')`

**Test each method here before moving to implementation file:**
- Create test credentials/environment variables
- Upload a sample DataFrame
- Download and verify structure matches

## 6. Tableau Integration Instructions

1. Open Tableau Desktop
2. Connect to data source: `data/processed/rfm_for_tableau.csv`
3. Create worksheets:
   - RFM Customer Segments (scatter plot: F vs M, colored by segment)
   - Customer Distribution by Segment (bar chart)
   - Conversion Funnel (funnel chart using funnel_metrics.csv)
4. Create dashboard combining all worksheets
5. Save as `dashboards/growth_dashboard.twb`

## 6a. Tableau Setup & Data Connection Guide

### Part 1: Download & Install Tableau Public (Free Version)

**Option A: Tableau Public (Cloud-based, Free Forever)**
1. Go to https://public.tableau.com/
2. Click **"Free Sign Up"** (top right)
3. Create account with email and password
4. Verify email
5. Download **"Tableau Public"** (desktop app)
   - Available for Windows and Mac
   - ~500MB download

**Option B: Tableau Desktop (Professional, 14-day Trial)**
1. Go to https://www.tableau.com/products/desktop/download
2. Enter email and choose platform
3. Download installer
4. Install and open - 14-day trial starts automatically
5. Great for testing before purchasing

### Part 2: Connect to CSV Data Source

**Steps to Connect in Tableau:**

1. **Open Tableau Public/Desktop**
2. Click **"Connect to Data"** (left panel)
3. Select **"Text File"** (for CSV) or **"Microsoft Excel"** (for xlsx)
4. Browse to your file: `data/processed/rfm_for_tableau.csv`
5. Click **"Open"**
6. **Data source panel** opens showing:
   - Column names
   - Data types (text, number, datetime)
   - Data preview at bottom
7. Click **"Sheet 1"** tab (bottom) to start creating visualization

### Part 3: Essential Tableau Concepts

| Concept | Meaning | Example |
|---------|---------|---------|
| **Dimension** | Categorical data (qualitative) | Segment, Region, Category |
| **Measure** | Numerical data (quantitative) | Revenue, Count, Average |
| **Rows/Columns** | Chart axes placement | Drag Segment to Rows, Revenue to Columns |
| **Marks** | Color, size, shape encoding | Color by Segment, Size by Revenue |
| **Filter** | Show only specific data | Show only "Champions" segment |
| **Aggregation** | How to sum data | SUM, AVG, COUNT, MIN, MAX |

### Part 4: Create Your First Dashboard

**Example 1: RFM Customer Segments Scatter Plot**

1. **Create new worksheet**
   - Click **"+"** tab → **"Sheet"**

2. **Drag fields:**
   - **Columns:** Drag `Frequency` (becomes x-axis)
   - **Rows:** Drag `Monetary` (becomes y-axis)
   - **Color:** Drag `Segment` (colors each point by segment)
   - **Size:** Drag `Customer_ID` count (bubble size by count)

3. **Result:** Scatter plot showing:
   - X-axis = Purchase frequency
   - Y-axis = Total spend
   - Color = Customer segment
   - Bubble size = Number of customers

**Example 2: Segment Distribution Bar Chart**

1. Create new sheet
2. **Drag fields:**
   - **Columns:** `Segment`
   - **Rows:** `Count of Customer_ID` (Tableau auto-counts)
   - **Color:** `Segment`

3. Result: Bar chart showing customer count per segment

**Example 3: Conversion Funnel**

1. Create new sheet
2. Load `funnel_metrics.csv` as new data source
3. **Drag fields:**
   - **Rows:** `Funnel_Step` (Visit → Cart → Checkout → Payment)
   - **Columns:** `User_Count`
   - Create bar chart with steps in order

### Part 5: Learning Resources

| Resource | Best For | Link |
|----------|----------|------|
| **Official Tableau Training** | Interactive videos & projects | https://www.tableau.com/learn/training |
| **Tableau YouTube Channel** | Quick tutorials (5-20 min) | https://www.youtube.com/user/tableaudesktop |
| **Tableau Public Gallery** | Real examples to explore | https://public.tableau.com/gallery |
| **Tableau Desktop Help** | Problem-solving & reference | Help menu → Tableau Desktop Help |
| **Tableau Community Forum** | Q&A with experts | https://community.tableau.com/ |

**Recommended Learning Path:**
1. **Start:** Watch "Getting Started with Tableau Desktop" (15 min)
2. **Practice:** Create scatter plot from your RFM data
3. **Explore:** Check Tableau Public Gallery for similar visualizations
4. **Deep Dive:** Official training on advanced topics (filters, parameters)

### Part 6: Connect to Data from S3 (Advanced)

**To load data directly from S3 (skip CSV download):**

1. In Tableau, click **"Connect to Data"**
2. Search for **"Amazon S3"** connector (may require download)
3. Enter:
   - AWS Access Key ID
   - AWS Secret Access Key
   - S3 bucket name
   - File path (e.g., `data/rfm_segments.csv`)
4. Click **"OK"** to load data
5. Continue with sheet creation

**Or use CSV approach:** Download from S3 first, then connect as Text File

### Part 7: Publishing Dashboards

**With Tableau Public (Free):**
1. Create your worksheets and dashboard
2. File → **"Save as"**
3. Sign in with Tableau Public credentials
4. Dashboard becomes public and shareable via URL
5. Others can view, download, or interact

**With Tableau Desktop:**
1. File → **"Save As"** to publish to Tableau Server/Tableau Online
2. Or export as PDF/Image for static sharing
